In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import matplotlib.pyplot as plt
import json
import os
from sklearn.model_selection import train_test_split

# --- 1. DATA PARSING & VOCAB ---
def get_word_indices(context, answer):
    context_words = context.split()
    answer_words = answer.split()
    try:
        start_idx = context_words.index(answer_words[0])
        end_idx = start_idx + len(answer_words) - 1
        return start_idx, end_idx
    except (ValueError, IndexError):
        return 0, 0

def parse_json_file(file_path):
    parsed_data = []
    if not os.path.exists(file_path): return parsed_data
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        for item in data['data']:
            for p in item['paragraphs']:
                context = p['context']
                for qa in p['qas']:
                    question = qa['question']
                    answer = qa['answers'][0]['text']
                    start, end = get_word_indices(context, answer)
                    parsed_data.append((question, context, answer, start, end))
    return parsed_data

class ArabicQAVocab:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<UNK>": 1}
        self.itos = {0: "<PAD>", 1: "<UNK>"}
        self.counter = 2
    
    def build_vocabulary(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.stoi:
                    self.stoi[word] = self.counter
                    self.itos[self.counter] = word
                    self.counter += 1

    def numericalize(self, text):
        return [self.stoi.get(w, self.stoi["<UNK>"]) for w in text.split()]

# --- 2. DATASET ---
class QADataset(Dataset):
    def __init__(self, data_list, vocab, max_len=50):
        self.data = data_list
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, index):
        q_txt, c_txt, _, s, e = self.data[index]
        q = self.vocab.numericalize(q_txt) + [0]*self.max_len
        c = self.vocab.numericalize(c_txt) + [0]*self.max_len
        return {
            "question": torch.tensor(q[:self.max_len]),
            "context": torch.tensor(c[:self.max_len]),
            "start_idx": torch.tensor(min(s, self.max_len-1)),
            "end_idx": torch.tensor(min(e, self.max_len-1))
        }

# --- 3. ARCHITECTURES ---
class RecurrentQA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 32)
        self.lstm = nn.LSTM(32, 64, batch_first=True, bidirectional=True)
        self.fc_start = nn.Linear(128, 50)
        self.fc_end = nn.Linear(128, 50)

    def forward(self, q, c):
        x, _ = self.lstm(self.emb(c))
        x = x.mean(dim=1) # Pooling sequence
        return self.fc_start(x), self.fc_end(x)

class TransformerQA(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(64, 4, batch_first=True), 1)
        self.fc_start = nn.Linear(64, 50)
        self.fc_end = nn.Linear(64, 50)

    def forward(self, q, c):
        x = self.enc(self.emb(c))
        x = x.mean(dim=1)
        return self.fc_start(x), self.fc_end(x)

# --- 4. TRAINING & METRICS ---
def train_and_eval(model, train_loader, val_loader, model_name):
    opt = optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    train_loss, val_loss = [], []
    
    for epoch in range(10):
        model.train()
        l = 0
        for b in train_loader:
            opt.zero_grad()
            s, e = model(b['question'], b['context'])
            loss = crit(s, b['start_idx']) + crit(e, b['end_idx'])
            loss.backward(); opt.step()
            l += loss.item()
        train_loss.append(l/len(train_loader))
        
        # Validation
        model.eval()
        v = 0
        with torch.no_grad():
            for b in val_loader:
                s, e = model(b['question'], b['context'])
                v += (crit(s, b['start_idx']) + crit(e, b['end_idx'])).item()
        val_loss.append(v/len(val_loader))
        
    plt.plot(train_loss, label='Train'); plt.plot(val_loss, label='Val'); plt.title(model_name); plt.legend(); plt.show()

# --- 5. EXECUTION ---
base_path = "/kaggle/input/datasets/ranaabdo/dataset2/"
files = [base_path + f for f in os.listdir(base_path)]
full_data = [item for f in files for item in parse_json_file(f)]
vocab = ArabicQAVocab()
vocab.build_vocabulary([i[0] for i in full_data] + [i[1] for i in full_data])

t_data, v_data = train_test_split(full_data, test_size=0.2)
train_loader = DataLoader(QADataset(t_data, vocab), batch_size=8)
val_loader = DataLoader(QADataset(v_data, vocab), batch_size=8)

train_and_eval(RecurrentQA(len(vocab.stoi)), train_loader, val_loader, "LSTM")
train_and_eval(TransformerQA(len(vocab.stoi)), train_loader, val_loader, "Transformer")